In [ ]:
!pip install -Uq transformers loguru

## Import

In [ ]:
import torch
from datetime import datetime
import re
import requests
import json
from loguru import logger
from kaggle_secrets import UserSecretsClient
from typing import Dict, Any


user_secrets = UserSecretsClient()
SEARCH_API_KEY = user_secrets.get_secret("SEARCH_API_KEY")


logger.add("agent.log", format="{time} {level} {message}")


logger.info("All moduels imported successfully!")

In [ ]:
sys_prompt = """You are a helpful and knowledgeable assistant. Your primary goal is to provide accurate and relevant information to the user.

Always think step-by-step and determine if any of your available tools can help fulfill the user's request. If a tool is relevant and can provide accurate information, use it.

Crucially, do not guess or fabricate information. If you cannot find a definitive answer using your tools or internal knowledge, clearly state that you don't know.
Be polite to the user. If you make mistake, Show a great apology.
Do not guess any answer.
Try to finish your response within 1000 tokens.
IMPORTANT: **You can call only a tool at a time.So, You MUST STOP after calling a tool*
When using any tool, act as a seamless assistant. Do not mention the names of the tools or their internal workings to the user. Integrate the tool's output naturally into your response. All responses should be in markdown format.
"""

## Tools

In [ ]:
supported_ext = {"json", "txt", "csv", "tsv", "py", "js", "c", "cpp", "ts", "html", "css", "log", "md"}

def read_file(path: str) -> Dict[str, str]:
    """Read the file and return the file content
    Args:
        path: file path to read. supported extension: ["json", "txt", "csv", "tsv", "py", "js", "c", "cpp", "ts", "html", "css", "log", "md]
    Returns:
        the file content (truncated to 300 lines)
    """
    ext = path.split(".")[-1].lower() if "." in path else ""
    if ext not in supported_ext:
        return {
            "status": "error",
            "error": f"Invalid file extension '{ext}'. Supported: {sorted(supported_ext)}"
        }

    if not os.path.exists(path):
        return {"status": "error", "error": "File Not Found"}

    if not os.path.isfile(path):
        return {"status": "error", "error": "Not a file"}

    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            lines = []
            for _ in range(300):
                line = f.readline()
                if not line:
                    break
                lines.append(line)
            content = "".join(lines)  # Keep original line endings
            return {"status": "success", "content": content}
    except Exception as e:
        return {"status": "error", "error": f"Read error: {str(e)}"}

In [ ]:
def write_file(file_path: str, file_content: str) -> str:
   """ Create and write the file with the filename
   Args:
       file_path: Path of thr file. Only the `data/` folder is writeable
       file_content: The file content
   """
   try:
      with open(file_path, "w") as f:
         f.write(file_content)
      return f"File saved as {file_path} successfully"

   except PermissionError:
      return "The directory isn't writable"

   except Exception as e:
      return f"Error: {e}"

In [ ]:
class Search:
    def __init__(self, api_key: str, top_k: int, location: str):
        self.api_key = api_key
        self.top_k = top_k
        self.location = location

    def search(self, query) -> dict:
        url = "https://api.scrapingdog.com/google/"
        params = {
            "api_key": self.api_key,
            "query": query,
            "results": self.top_k,
            "country": self.location,
            "page": '0'
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()

            res =  response.json()
            res["status"] = "success"
            return res
        except Exception as e:
            return {"status": "error", "error": str(e)}

    def __call__(self, query: str) -> dict:
        data = self.search(query)
        # if error hit, .search returns error message return the error data
        if data.get("status") == "error":
            return data
        output = {
            "status": "success",
            "results": [],
            "answer": None
        }

        answers = data.get("answer_box", None)
        if isinstance(answers, dict):
            answers = answers.get("answer", "")
            if answers: output["answer"] = answers

        organic_results = data.get("organic_results", None)
        if isinstance(organic_results, list):
            for result in organic_results:
                title = result.get("title", "")
                snippet = result.get("snippet", "")
                link = result.get("link", "")
                output["results"].append({
                   "title": title,
                   "snippet": snippet,
                   "link": link
                })

        return output


def search_web(query: str, top_k: int=5, location: str="us") -> dict:
    """
    Search the web and get the search data
    Args:
        query: The search query string
        top_k: Number of top results to return
        location: Search region for personalization (e.g., "us") (optional).
    Returns:
        The search data as json
    """
    sr = Search(SEARCH_API_KEY, top_k, location)
    return sr(query)

In [ ]:
def get_time() -> str:
  """Get the current datetime"""
  time = datetime.now()
  return time.strftime("%d-%m-%Y %H:%I %p")

tools = [get_time, search_web, read_file, write_file]

# Multimodal model

In [ ]:
import os

os.makedirs("data", exist_ok=True)

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig, TextStreamer

model_id = "Qwen/Qwen3.5-9B"

#q_cfg = BitsAndBytesConfig(
#   load_in_4bit=True,
#   bnb_4bit_compute_dtype=torch.bfloat16
#)

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto",
    #quantization_config=q_cfg
)

logger.info("Model loaded")

In [ ]:
class Agent:
    def __init__(self, model, processor, tools, sys_prompt, *, temperature=1.0, top_p=0.9, top_k=30):
        self.model = model
        self.processor = processor
        self.tool_map = {tool.__name__: tool for tool in tools}
        self.tools = tools
        self.msg = []
        self.sys_prompt = sys_prompt
        self.temperature = temperature
        self.top_k = top_k
        self.top_p = top_p


    def _parse_tool_call(self, text):
        tool_match = re.search(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL)
        if not tool_match:
            return None

        block = tool_match.group(1)

        func = re.search(r'<function=(.*?)>', block)
        if not func:
            return None

        name = func.group(1).strip()

        params = {}
        for p, v in re.findall(r'<parameter=(.*?)>\s*(.*?)\s*</parameter>', block, re.DOTALL):
            params[p.strip()] = v.strip()

        return {"name": name, "arguments": params}

    def _strip_think(self, out):
        return re.sub(r"(?s).*?</think>\s*", "", out, count=1)

    def _run(self, think, max_response_len=512, max_steps=2):
        for _ in range(max_steps):
            inputs = self.processor.apply_chat_template(
              self.msg,
              tokenize=True,
              add_generation_prompt=True,
              return_tensors="pt",
              return_dict=True,
              enable_thinking=think,
              tools=self.tools
            ).to("cuda")

            streamer = TextStreamer(self.processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
            inp_len = inputs["input_ids"].shape[1]
            output = self.model.generate(
                **inputs,
                do_sample=True,
                temperature=self.temperature,
                max_new_tokens=max_response_len,
                streamer=streamer,
                top_p=self.top_p,
                top_k=self.top_k,
                use_cache=True
            )
            out =  self.processor.decode(output[0][inp_len:])
            out = self._strip_think(out).strip()
            tool_call = self._parse_tool_call(out)

            if tool_call:
                    fn_name = tool_call["name"]
                    args = tool_call["arguments"]
                    try:
                        if fn_name in self.tool_map:
                          tool_result = self.tool_map[fn_name](**args)
                          logger.info(f"Used {fn_name} with {args}")
                        else:
                            tool_result = f"No tool named: {fn_name}. Make sure you called the right tool"
                            logger.info(f"Agent used wrong tool: {fn_name}")
                    except Exception as e:
                      tool_result = f"An error occured when calling the tool: {fn_name}. Error: {e}"
                      logger.debug(f"Error: {fn_name} args: {args}")

                    self.msg.append({
                        "role": "assistant",
                        "content": [{"type": "text", "text": out}]
                    })
                    self.msg.append({
                        "role": "tool",
                        "name": fn_name,
                        "content": [{"type": "text", "text": tool_result}]
                    })
            else:
                self.msg.append({
                    "role": "assistant",
                    "content": [{"type": "text", "text": out}]
                })
                break

    def chat(self, max_response_len=512, max_steps=2):
        while True:
            if not self.msg:
                self.msg.append({"role": "system", "content": self.sys_prompt})

            prompt = input("Your prompt: ")
            if prompt == "break": break
            img = input("Enter image(url or file path) - Optional: ")
            think = input("Enable thinking (Y/N): ").strip().lower() == "y"

            user_content = [{
                  "type": "text",
                  "text": prompt
            }]
            if img:
                user_content.append({"type": "image", "image": img})

            self.msg.append({
               "role": "user",
               "content": user_content
            })
            self._run(think, max_response_len, max_steps)

    def save_chat(self, file_path):
        with open(file_path, "w") as f:
            json.dump(self.msg, f, indent=2)
            logger.info(f"Chat saved as {file_path}")

    def load_chat_history(self, file_path):
        if os.path.exists(file_path):
            with open(file_path) as f:
                history = f.read()
                self.msg = json.loads(history)
                logger.info("Chat loaded successfully")
        else: print("File Not Found")

In [ ]:
agent = Agent(
  model,
  processor,
  tools=tools,
  sys_prompt=sys_prompt,
  temperature=0.8
)

In [ ]:
agent.chat(max_response_len=2056)

In [ ]:
agent.save_chat("chat.json")

In [ ]:
from IPython.display import display, Markdown
with open("data/eid.md") as f:
    content = f.read()

display(Markdown(content))